# Medical Insurance Cost Prediction — Multiple Linear Regression

**Dataset:** [Kaggle - Medical Cost Personal Insurance](https://www.kaggle.com/datasets/mirichoi0218/insurance)

**Goal:** Predict insurance `charges` from personal/health features using Multiple Linear Regression.

This notebook is organized by assignment task. Each task is its own section, so you can
extend it (Task 3, Task 4, ...) by adding new sections below, and committing after each one.

## Task 1: Data Understanding

1. Load the dataset using Pandas.
2. Display the first five records.
3. Identify numerical features, categorical features, and the target variable.

In [2]:
import pandas as pd

# 1. Load the dataset
df = pd.read_csv("data/insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [4]:
df.shape

(1338, 7)

### Identify feature types

In [4]:
numerical_features = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = df.select_dtypes(include=["object", "str"]).columns.tolist()

target_variable = "charges"
numerical_features.remove(target_variable)

print("Numerical features   :", numerical_features)
print("Categorical features  :", categorical_features)
print("Target variable       :", target_variable)

Numerical features   : ['age', 'bmi', 'children']
Categorical features  : ['sex', 'smoker', 'region']
Target variable       : charges


## Task 2: Data Preprocessing (2 Marks)

1. Check for missing values.
2. Encode categorical variables (`sex`, `smoker`, `region`).
3. Split the dataset into 80% training and 20% testing.

### 1. Check for missing values

In [5]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [6]:
print("Total missing values:", df.isnull().sum().sum())

Total missing values: 0


### 2. Encode categorical variables

- `sex` and `smoker` are binary → label-encoded as 0/1.
- `region` has 4 categories → one-hot encoded with `drop_first=True` to avoid the
  dummy-variable trap (multicollinearity).

In [7]:
df_encoded = df.copy()

df_encoded["sex"] = df_encoded["sex"].map({"female": 0, "male": 1})
df_encoded["smoker"] = df_encoded["smoker"].map({"no": 0, "yes": 1})

df_encoded = pd.get_dummies(df_encoded, columns=["region"], drop_first=True)

df_encoded.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,False,False,True
1,18,1,33.770,1,0,1725.55230,False,True,False
2,28,1,33.000,3,0,4449.46200,False,True,False
3,33,1,22.705,0,0,21984.47061,True,False,False
4,32,1,28.880,0,0,3866.85520,True,False,False


### 3. Train/test split (80/20)

In [8]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=["charges"])   # features
y = df_encoded["charges"]                  # target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

X_train shape: (1070, 8)
X_test shape : (268, 8)
y_train shape: (1070,)
y_test shape : (268,)


In [9]:
# Save processed splits for use in later tasks (model building/evaluation)
X_train.to_csv("data/X_train.csv", index=False)
X_test.to_csv("data/X_test.csv", index=False)
y_train.to_csv("data/y_train.csv", index=False)
y_test.to_csv("data/y_test.csv", index=False)

print("Saved train/test splits to data/")

Saved train/test splits to data/


---
## Task 3: (add here)

Build a Multiple Linear Regression model using: age, sex, bmi, children, smoker, region as
features and charges as the target.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Intercept:", model.intercept_)

In [ ]:
coeff_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_
}).sort_values(by="Coefficient", key=abs, ascending=False)
coeff_df

In [ ]:
y_pred = model.predict(X_test)
results_df = pd.DataFrame({"Actual": y_test.values, "Predicted": y_pred})
results_df.head(10)

##Task - 4

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"R2 : {r2:.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, edgecolor="k")
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")
plt.xlabel("Actual Charges"); plt.ylabel("Predicted Charges")
plt.title("Actual vs Predicted Insurance Charges")
plt.savefig("actual_vs_predicted.png", dpi=150)
plt.show()